# Section 4b: Logistic regression and Bi-LSTM compared

Compares four toxicity models on the pooled validation split and zero-shot on r/jobs comments:

| Model | Trained and saved by |
|---|---|
| `LR (standard)`: TF-IDF logistic regression | `3_pooled_data.ipynb` |
| `LR (balanced)`: TF-IDF logistic regression with balanced class weights | `3_pooled_data.ipynb` |
| `LSTM (standard)`: Bi-LSTM without class weights | `4_LSTM_model.ipynb` |
| `LSTM (balanced)`: Bi-LSTM with balanced class weights | `4_LSTM_model.ipynb` |

Nothing is trained or tuned here. The notebook loads the saved models and scores them, all at the fixed threshold of 0.5.

Run `3_pooled_data.ipynb` and `4_LSTM_model.ipynb` first. They must have saved:
- `dataset/pooled/pooled_train.csv`, `pooled_val.csv` and `manifest.json` (notebook 3)
- `models/baselines/` with the two logistic regressions and `baselines_manifest.json` (notebook 3)
- `models/bayes_search_pooled_unweighted/best_kfold/` and `models/bayes_search_pooled_balanced/best_kfold/` (notebook 4)
- `models/tokenizer/` (notebook 4; if it is missing, the tokenizer is refitted on the training text with the same settings)

Pipeline: load the pooled data → load the four models → compare them on the validation split → zero-shot test on r/jobs.

## 1. Setup
Finds the project folder (so `functions/` can be imported and the relative paths work), then imports the libraries and the shared project code.

In [ ]:
import sys
import os
import json
import hashlib
import warnings
from pathlib import Path

# work from this notebook's folder: find the project root and add it to Python's search path
here = Path.cwd()
root = next((p for p in (here, here.parent) if (p / 'functions' / 'preprocessing.py').exists()), None)
if root is None:
    raise RuntimeError('Open this notebook from the project folder (the one with functions/) or from its notebook/ folder.')
sys.path.insert(0, str(root))
os.chdir(root / 'notebook')

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from sklearn.metrics import average_precision_score, confusion_matrix, ConfusionMatrixDisplay, precision_recall_curve
from tensorflow.keras.preprocessing.text import Tokenizer, tokenizer_from_json
from tensorflow.keras.preprocessing.sequence import pad_sequences

from functions.preprocessing import cleanup_text
from functions.cv import kfold_wrapper
from functions.metrics import paired_bootstrap, toxicity_metrics

# the nightly Keras build wrongly warns that LSTM / Bidirectional destroy the mask (masking works)
warnings.filterwarnings('ignore', message='.*was passed an input with a mask attached.*')

## 2. Load the pooled data
Reads the fixed train / validation splits written by `3_pooled_data.ipynb`. The validation comments are what every model is scored on in section 4, and the training comments are only needed to rebuild the tokenizer if none was saved. The files are checked against `manifest.json`, so the notebook stops if they were edited or regenerated differently.

In [ ]:
pooled_dir = '../dataset/pooled'
train_path = os.path.join(pooled_dir, 'pooled_train.csv')
val_path = os.path.join(pooled_dir, 'pooled_val.csv')
manifest_path = os.path.join(pooled_dir, 'manifest.json')
for path in (train_path, val_path, manifest_path):
    if not os.path.exists(path):
        raise FileNotFoundError(f'{path} not found. Run notebook/3_pooled_data.ipynb first to create the pooled splits.')

def read_pooled(path):
    # orig_id must stay a string and empty strings must not become NaN
    return pd.read_csv(path, dtype={'orig_id': str}, keep_default_na=False)

train_df, val_df = read_pooled(train_path), read_pooled(val_path)

# the splits must be the ones recorded in the manifest (data, mapping or cleaning may have changed since)
with open(manifest_path, encoding='utf-8') as f:
    manifest = json.load(f)
for name, frame in [('train', train_df), ('val', val_df)]:
    digest = hashlib.sha256(frame.to_csv(index=False, lineterminator='\n').encode()).hexdigest()
    if digest != manifest['split_sha256'][name]:
        raise ValueError(f'pooled_{name}.csv does not match manifest.json: regenerate the splits with notebook 3.')

y_val = val_df['label'].to_numpy(dtype=int)
print(f'Validation split: {len(val_df):,} comments, toxic share {y_val.mean():.3f}')

## 3. Load the trained models
Loads the four saved models and the tokenizer.

- **Logistic regressions:** the two TF-IDF pipelines saved by notebook 3 take the cleaned text directly.
- **Bi-LSTMs:** the best model of each search saved by notebook 4. Text goes through the same steps as in training: `cleanup_text` → the training tokenizer → pad or cut at the end to `max_length` → give an empty row the unknown-word id. The saved tokenizer is loaded if notebook 4 saved one; otherwise it is refitted on the training text with the same settings, which gives the same result.
- **Checks:** each saved model must have been trained on the training split in `manifest.json`, and each Bi-LSTM must have the class weights its name says, so a stale or swapped folder stops the notebook.

Every model uses the same fixed threshold of 0.5. `predict_proba(name, texts)` returns the probability of toxic for cleaned comments.

In [ ]:
THRESHOLD = 0.5  # fixed decision threshold for every model
VOCAB_SIZE, MAX_LENGTH = 10000, 80  # as in notebook 4

# --- tokenizer
tokenizer_dir = '../models/tokenizer'
tokenizer_path = os.path.join(tokenizer_dir, 'tokenizer.json')
if os.path.exists(tokenizer_path):
    with open(tokenizer_path, encoding='utf-8') as f:
        tokenizer = tokenizer_from_json(f.read())
    with open(os.path.join(tokenizer_dir, 'settings.json'), encoding='utf-8') as f:
        tokenizer_settings = json.load(f)
    if tokenizer_settings['train_split_sha256'] != manifest['split_sha256']['train']:
        raise ValueError('The saved tokenizer was fitted on a different training split: re-run notebook 4.')
    if (tokenizer_settings['vocab_size'], tokenizer_settings['max_length']) != (VOCAB_SIZE, MAX_LENGTH):
        raise ValueError('The saved tokenizer settings differ from VOCAB_SIZE and MAX_LENGTH above.')
    print('Loaded the tokenizer saved by notebook 4')
else:
    print('No saved tokenizer found: fitting one on the training text with the settings of notebook 4')
    tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
    tokenizer.fit_on_texts(train_df['text'])

def encode(texts):
    # tokenize, pad or cut at the end, and give an all-padding row the unknown-word id (as in notebook 4)
    X = pad_sequences(tokenizer.texts_to_sequences(texts), maxlen=MAX_LENGTH, padding='post', truncating='post')
    X[(X == 0).all(axis=1), 0] = tokenizer.word_index[tokenizer.oov_token]
    return X

# --- logistic regressions saved by notebook 3
baseline_dir = '../models/baselines'
manifest_file = os.path.join(baseline_dir, 'baselines_manifest.json')
if not os.path.exists(manifest_file):
    raise FileNotFoundError(f'{manifest_file} not found. Run notebook/3_pooled_data.ipynb first (it saves the fitted baselines).')
with open(manifest_file, encoding='utf-8') as f:
    baselines_manifest = json.load(f)
if baselines_manifest['train_split_sha256'] != manifest['split_sha256']['train']:
    raise ValueError('The saved logistic regressions were fitted on a different training split: re-run notebook 3.')
logreg = {
    'LR (standard)': joblib.load(os.path.join(baseline_dir, 'tfidf_logreg.joblib')),
    'LR (balanced)': joblib.load(os.path.join(baseline_dir, 'tfidf_logreg_balanced.joblib')),
}

# --- Bi-LSTMs saved by notebook 4: the best model of each search
lstm_dirs = {
    'LSTM (standard)': '../models/bayes_search_pooled_unweighted/best_kfold',
    'LSTM (balanced)': '../models/bayes_search_pooled_balanced/best_kfold',
}
expected_class_weight = {'LSTM (standard)': None, 'LSTM (balanced)': 'balanced'}
lstm = {}
for name, directory in lstm_dirs.items():
    if not os.path.exists(os.path.join(directory, 'state.pkl')):
        raise FileNotFoundError(f'{directory} not found. Run notebook/4_LSTM_model.ipynb first (the search saves its best model there).')
    lstm[name] = kfold_wrapper.load(directory)
    trained_with = getattr(lstm[name], 'class_weight', None)  # results saved before the option existed had no class weights
    if trained_with != expected_class_weight[name]:
        raise ValueError(f'{name}: the saved model was trained with class_weight={trained_with!r}, expected {expected_class_weight[name]!r}.')

model_names = list(logreg) + list(lstm)
model_colors = {'LR (standard)': '#7f7f7f', 'LR (balanced)': '#8172B3', 'LSTM (standard)': '#55A868', 'LSTM (balanced)': '#4C72B0'}

def predict_proba(name, texts):
    # probability of toxic for already cleaned comments
    texts = list(texts)
    if name in logreg:
        return logreg[name].predict_proba(texts)[:, 1]
    return lstm[name].final_model.predict(encode(texts), verbose=0).ravel()

print('Models loaded:', ', '.join(model_names))

### Evaluation helpers
`evaluate` returns the confusion counts and the proposal's metrics for one model at the fixed threshold: precision, recall (the true positive rate, TPR), the false positive rate (FPR), macro F1, PR-AUC (average precision) and ROC-AUC. Recall and FPR are each computed within a single class, so they do not depend on the share of toxic comments; precision, macro F1 and PR-AUC do. The plotting helpers draw one confusion matrix per model in a grid, and the models' precision–recall curves with their operating points on one set of axes.

In [ ]:
def evaluate(y, proba, threshold=THRESHOLD):
    pred = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    m = toxicity_metrics(y, proba, threshold)
    return {'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn, 'precision': m['precision'], 'recall (TPR)': m['recall'],
            'FPR': fp / (fp + tn), 'macro_f1': m['macro_f1'], 'pr_auc': m['pr_auc'], 'roc_auc': m['roc_auc']}

def results_table(y, probas):
    return pd.DataFrame({name: evaluate(y, p) for name, p in probas.items()}).T.infer_objects()

def plot_confusion_matrices(y, probas, suptitle):
    fig, axes = plt.subplots(2, 2, figsize=(9, 8))
    for ax, (name, p) in zip(axes.ravel(), probas.items()):
        ConfusionMatrixDisplay.from_predictions(
            y, (p >= THRESHOLD).astype(int), labels=[0, 1], display_labels=['Non-toxic', 'Toxic'],
            cmap='Blues', colorbar=False, ax=ax)
        ax.set_title(name)
    fig.suptitle(suptitle)
    fig.tight_layout()
    plt.show()

def plot_pr_curves(y, probas, title):
    fig, ax = plt.subplots(figsize=(7, 5.5))
    for name, p in probas.items():
        precision, recall, _ = precision_recall_curve(y, p)
        ax.step(recall, precision, where='post', color=model_colors[name], linewidth=2,
                label=f'{name} (PR-AUC = {average_precision_score(y, p):.3f})')
        pred = (p >= THRESHOLD).astype(int)
        ax.scatter([((pred == 1) & (y == 1)).sum() / (y == 1).sum()], [((pred == 1) & (y == 1)).sum() / max((pred == 1).sum(), 1)],
                   color=model_colors[name], edgecolors='black', zorder=3, s=70)
    ax.axhline(y.mean(), color='grey', linestyle='--', label=f'Chance (PR-AUC = {y.mean():.3f})')
    ax.set(xlabel='Recall', ylabel='Precision', title=title, xlim=(0, 1.02), ylim=(0, 1.05))
    ax.legend(loc='lower left', fontsize=8)
    plt.tight_layout()
    plt.show()

## 4. Validation performance
Scores the pooled validation split with all four models. The validation comments come from the same four sources as the training data, so these numbers are in-domain and optimistic about r/jobs. The table gives the confusion counts and metrics at the fixed 0.5 threshold; the matrices show how many toxic comments each model catches and how many clean ones it flags; and the curves rank the models independently of any threshold, with a dot at each model's 0.5 operating point. The validation split has about 730 toxic comments, so differences of a point or two between models are within noise.

In [ ]:
val_proba = {name: predict_proba(name, val_df['text']) for name in model_names}
val_results = results_table(y_val, val_proba)
print(f'Validation split: {len(y_val):,} comments, toxic share {y_val.mean():.1%}, threshold {THRESHOLD:g}')
display(val_results.round(3))

plot_confusion_matrices(y_val, val_proba, 'Validation confusion matrices')
plot_pr_curves(y_val, val_proba, 'Validation precision-recall curves')

## 5. Zero-shot validation on r/jobs
Evaluates the four models on the shared 100-comment r/jobs sample: **50 non-toxic (`0`) and 50 toxic (`1`) AI labels**. These are toxicity labels, not sentiment. AI annotations are the reference labels for this test set and are used directly for evaluation, so the scores measure **agreement with AI annotations**; the label source and the individual rationales remain available with the data.

None of the models was trained, tuned or thresholded on this sample: each is applied as saved, at the fixed 0.5 threshold. The sample is balanced and keyword-enriched, with some comments from the same threads, so its precision, macro F1 and PR-AUC do not estimate live r/jobs performance or toxicity prevalence. Recall and the false positive rate do not depend on the toxic share, so they are the measures to compare with the validation split, but each rests on only 50 comments, and differences of a few points are within noise. Provenance and per-comment label rationales are in `../dataset/reddit/`.

### Load and check the sample
Keep the source text intact. Stop on missing labels, duplicate comments, an unexpected class balance or a comment without a completed AI annotation, instead of silently treating missing labels as non-toxic.

In [ ]:
reddit_path = Path('../dataset/reddit/test_100_ai_labeled.csv')
reddit_df = pd.read_csv(reddit_path, dtype=str, keep_default_na=False)
reddit_required = {'comment_id', 'text', 'subreddit', 'is_toxic', 'permalink',
                   'annotation_status', 'annotator', 'annotation_notes'}
if not reddit_required.issubset(reddit_df.columns):
    raise ValueError(f'Missing Reddit columns: {sorted(reddit_required - set(reddit_df.columns))}')
if len(reddit_df) != 100 or reddit_df['is_toxic'].value_counts().to_dict() != {'0': 50, '1': 50}:
    raise ValueError('Expected exactly 100 labelled comments: 50 non-toxic (0), 50 toxic (1).')
if (reddit_df['comment_id'].str.strip().eq('').any()
        or reddit_df['comment_id'].duplicated().any()
        or reddit_df['text'].str.strip().eq('').any()):
    raise ValueError('Reddit IDs must be nonempty and unique; comment text must be nonempty.')
reddit_text_keys = reddit_df['text'].str.normalize('NFKC').str.casefold().str.split().str.join(' ')
if reddit_text_keys.duplicated().any() or reddit_text_keys.isin(['[deleted]', '[removed]']).any():
    raise ValueError('Duplicate or deleted/removed Reddit comments cannot be evaluated.')
if not reddit_df['subreddit'].eq('jobs').all():
    raise ValueError('This evaluation expects r/jobs comments only.')
if not reddit_df['annotation_status'].eq('ai_labeled').all():
    raise ValueError('Every test comment must have a completed AI annotation (ai_labeled).')
reddit_annotators = reddit_df['annotator'].str.strip()
if (not reddit_annotators.str.casefold().str.startswith('ai:').all()
        or reddit_annotators.str[3:].str.strip().eq('').any()
        or reddit_df['annotation_notes'].str.strip().eq('').any()):
    raise ValueError('Every AI-labelled comment needs an AI:<model> source and an annotation reason.')

reddit_label_source = 'AI annotations'
print(f'Loaded {len(reddit_df)} r/jobs comments from {reddit_path}')
print(f'Reference: {reddit_label_source}; scores measure agreement with these labels.')
display(reddit_df.groupby('is_toxic').size().rename(index={'0': 'Non-toxic', '1': 'Toxic'}).to_frame('count'))

### Apply the training pipeline and score the sample
Lowercase → `cleanup_text` gives the cleaned text that every model expects; the Bi-LSTMs then add the tokenizer and padding inside `predict_proba`. The truncation count shows how many comments exceed the Bi-LSTMs' input length of 80 tokens (the logistic regressions see the whole comment); it is diagnostic, not a reason to change the trained configuration.

In [ ]:
reddit_dependencies = ['model_names', 'predict_proba', 'tokenizer', 'cleanup_text', 'MAX_LENGTH', 'THRESHOLD']
reddit_missing = [name for name in reddit_dependencies if name not in globals()]
if reddit_missing:
    raise RuntimeError(f'Run the setup and model-loading sections first; missing: {reddit_missing}')

reddit_cleaned = [cleanup_text(text.lower()) for text in reddit_df['text']]
y_reddit = reddit_df['is_toxic'].astype(int).to_numpy()
reddit_n_tokens = np.array([len(sequence) for sequence in tokenizer.texts_to_sequences(reddit_cleaned)])

reddit_proba = {}
for name in model_names:
    proba = np.asarray(predict_proba(name, reddit_cleaned), dtype=float)
    if proba.shape != y_reddit.shape or not np.isfinite(proba).all() or ((proba < 0) | (proba > 1)).any():
        raise ValueError(f'{name} must return one finite toxicity probability in [0, 1] per comment.')
    reddit_proba[name] = proba

print(f'Comments longer than {MAX_LENGTH} tokens (cut for the Bi-LSTMs): {(reddit_n_tokens > MAX_LENGTH).sum()}/{len(reddit_df)}')
print(f'Comments with no tokens after preprocessing: {(reddit_n_tokens == 0).sum()}')

### Metrics, confusion matrices and precision–recall curves
Report the proposal's precision and recall for the **toxic class**, macro F1 over both classes, and **PR-AUC as average precision**, as in section 4, with the false positive rate next to recall. The curves describe ranking; do not choose a new threshold from this sample. The Reddit sample is 50% toxic, so its precision, macro F1 and PR-AUC are not comparable with the validation split (15% toxic); use recall and FPR for that.

In [ ]:
reddit_results = results_table(y_reddit, reddit_proba)
print(f'Reddit sample: {len(y_reddit)} comments, toxic share {y_reddit.mean():.0%}, threshold {THRESHOLD:g} | reference: {reddit_label_source}')
display(reddit_results.round(3))

plot_confusion_matrices(y_reddit, reddit_proba, f'Reddit confusion matrices (reference: {reddit_label_source})')
plot_pr_curves(y_reddit, reddit_proba, 'Reddit precision-recall curves')

### Validation against r/jobs
Puts recall and the false positive rate of each model side by side for the two datasets, which are the two measures that do not depend on the share of toxic comments. A model that transfers well keeps both close to its validation values. The change column is r/jobs minus validation.

In [ ]:
shift = pd.concat({
    'validation': val_results[['recall (TPR)', 'FPR']],
    'r/jobs sample': reddit_results[['recall (TPR)', 'FPR']],
    'change': reddit_results[['recall (TPR)', 'FPR']] - val_results[['recall (TPR)', 'FPR']],
}, axis=1)
display(shift.astype(float).round(3))

### Precision at other shares of toxic comments
Precision depends on how common toxic comments are, recall does not. For every share of toxic comments from 5% to 90% in steps of 5%, each curve starts from a model's confusion matrix at the 0.5 threshold and changes only the number of toxic comments: TP and FN are scaled, while TN and FP stay fixed. Recall and the false positive rate therefore stay the same, and only the precision changes. There is one panel per dataset with all four models on the same axes: the left panel uses the validation rates, which are stable but in-domain; the right panel uses the r/jobs sample rates, which show the models under domain shift but rest on 50 comments per class, keyword-selected, with AI labels, so treat it as a pessimistic, uncertain case.

The curves are not straight lines: precision rises fast while toxic comments are rare and flattens as they become common. The dashed line is what flagging at random would give. The circles mark the share of toxic comments each dataset actually has (about 15% for validation, 50% for the sample). The share on r/jobs is unknown, so read the curves as an illustration, not a forecast.

In [ ]:
def precision_by_share(tp0, fn0, fp0, tn0, shares):
    precision = []
    for share in shares:
        # factor that gives this share of toxic comments when TN and FP are kept
        scale = share / (1 - share) * (tn0 + fp0) / (tp0 + fn0)
        precision.append(tp0 * scale / (tp0 * scale + fp0))
    return np.array(precision)

shares = np.arange(0.05, 0.90 + 1e-9, 0.05)
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, (title, results) in zip(axes, [('Validation rates', val_results), ('r/jobs sample rates', reddit_results)]):
    for name in model_names:
        r = results.loc[name]
        tp, fn, fp, tn = (float(r[k]) for k in ('TP', 'FN', 'FP', 'TN'))
        color = model_colors[name]
        ax.plot(shares, precision_by_share(tp, fn, fp, tn, shares), 'o-', color=color, markersize=3,
                label=f"{name} (recall {tp / (tp + fn):.2f}, FPR {fp / (fp + tn):.2f})")
        # the share of toxic comments this dataset actually has
        ax.scatter([(tp + fn) / (tp + fn + fp + tn)], [tp / (tp + fp)], s=130, facecolors='none',
                   edgecolors=color, linewidths=2, zorder=3)
    ax.plot([0, 1], [0, 1], color='grey', linestyle='--', label='Random flagging (precision = share toxic)')
    ax.set(xlim=(0, 1), ylim=(0, 1.02), xlabel='Share of toxic comments', title=title)
    ax.xaxis.set_major_formatter(PercentFormatter(1.0))
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    ax.legend(loc='lower right', fontsize=8)
axes[0].set_ylabel('Precision')
fig.suptitle('Precision against share of toxic comments (recall and FPR held fixed)')
plt.tight_layout()
plt.show()

### Are the gaps larger than chance?
The tables above give one number per model, but a few hundred or a hundred comments leave room for luck. A paired bootstrap shows how much a gap between two models could move. Each round draws the comments again with replacement, the same comments for both models, recomputes the metrics and takes the difference, model a minus model b. After 1,000 rounds, the middle 95% of the differences is the interval below. If it includes zero, this sample cannot tell the two models apart on that metric. The differences and intervals are in percentage points.

The comparisons are the two Bi-LSTMs against the balanced logistic regression, the model we would deploy. Treat the intervals as a guide, not a test: they assume the comments are independent, which is not quite true for r/jobs, where some comments come from the same threads, and nothing here is used to choose a model or a threshold. The r/jobs intervals are wide because there are only 100 comments. The seed is fixed, so the numbers repeat exactly.

In [ ]:
BOOTSTRAP = {'n_boot': 1000, 'seed': 0}  # number of resamples and the random seed
comparisons = [('LSTM (standard)', 'LR (balanced)'), ('LSTM (balanced)', 'LR (balanced)')]

bootstrap = {}
for dataset, y, probas in [('validation', y_val, val_proba), ('r/jobs sample', y_reddit, reddit_proba)]:
    for a, b in comparisons:
        bootstrap[(dataset, f'{a} minus {b}')] = paired_bootstrap(y, probas[a], probas[b], threshold=THRESHOLD, **BOOTSTRAP)
bootstrap = pd.concat(bootstrap, names=['dataset', 'comparison'])

bootstrap_points = (bootstrap[['difference', 'ci_low', 'ci_high']] * 100).round(1).rename(columns={
    'difference': 'observed difference on the full set',
    'ci_low': 'bootstrap 2.5th percentile (95% interval lower bound)',
    'ci_high': 'bootstrap 97.5th percentile (95% interval upper bound)'})
bootstrap_points['includes zero'] = (bootstrap['ci_low'] <= 0) & (bootstrap['ci_high'] >= 0)
print(f"{BOOTSTRAP['n_boot']} resamples, seed {BOOTSTRAP['seed']}; differences in percentage points (a minus b)")
display(bootstrap_points)

### Inspect disagreements with the reference labels
`reddit_predictions` retains all 100 predictions with original text, cleaned input, label status and source links, plus each model's probability and error type. `reddit_errors` contains the comments on which at least one model disagrees with the reference label, with false positives and negatives defined relative to the supplied labels. For this AI-labelled sample, these may be annotation errors as well as model errors. Inspect the sources before drawing conclusions; keep this set out of model and threshold selection.

In [ ]:
reddit_predictions = reddit_df.copy()
reddit_predictions['reference_label'] = y_reddit
for name in model_names:
    pred = (reddit_proba[name] >= THRESHOLD).astype(int)
    reddit_predictions[f'p_{name}'] = reddit_proba[name]
    reddit_predictions[f'error_{name}'] = np.select(
        [(pred == 1) & (y_reddit == 0), (pred == 0) & (y_reddit == 1)],
        ['FP', 'FN'], default='ok')
reddit_predictions['cleaned_text'] = reddit_cleaned
reddit_predictions['n_tokens'] = reddit_n_tokens

error_columns = [f'error_{name}' for name in model_names]
reddit_errors = reddit_predictions.loc[(reddit_predictions[error_columns] != 'ok').any(axis=1)].copy()
reddit_errors['n_models_wrong'] = (reddit_errors[error_columns] != 'ok').sum(axis=1)
reddit_errors = reddit_errors.sort_values(['n_models_wrong', 'reference_label'], ascending=False)

print(f'Comments with at least one disagreement: {len(reddit_errors)}/{len(reddit_predictions)} against {reddit_label_source}')
print('Errors per model (FN = missed toxic, FP = false alarm):')
display(pd.DataFrame({name: reddit_predictions[f'error_{name}'].value_counts() for name in model_names})
        .reindex(['FN', 'FP', 'ok']).fillna(0).astype(int))
print(f'Wrong in all {len(model_names)} models: {(reddit_errors["n_models_wrong"] == len(model_names)).sum()}')
shown = (['comment_id', 'reference_label', 'n_models_wrong']
         + [col for name in model_names for col in (f'p_{name}', f'error_{name}')]
         + ['n_tokens', 'text', 'cleaned_text', 'permalink'])
with pd.option_context('display.max_colwidth', 180):
    display(reddit_errors[shown])